# Processamento de Variáveis Socioeconômicas - Desafio 2

## ZettaLab - Ciência e Governança de Dados

**Objetivo**: Processar dados do IBGE SIDRA para criar variáveis preditoras do modelo de evasão/reprovação escolar.

**Metodologia**: CRISP-DM (Fase 3 - Preparação dos Dados)

**Período**: 2018-2022

**Granularidade**: UF (27 estados)

---

## Variáveis a Processar

| # | Variável | Fonte | Tabela SIDRA |
|---|----------|-------|-------------|
| 1 | Índice de Gini | IBGE/PNAD | 7435 |
| 2 | Taxa de Gravidez Adolescente | IBGE/Registro Civil | 2609 |
| 3 | PIB Total | IBGE/SCR | 5938 |

## Decisões de Projeto

- **Descartada**: Taxa de Analfabetismo (dados faltantes 2020-2021, interpolação inadequada devido à pandemia)
- **Descartada**: Anos de Estudo (estrutura incorreta nos dados baixados)
- **PIB**: Usado como valor total (Mil R$), não per capita (já temos Renda Per Capita)

---

## 1. Setup e Importações

In [ ]:
import pandas as pd
import numpy as np
import os

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Caminhos dos diretórios
BASE_DIR = '/home/teodoro/Documents/ZettaLab/ZettaLab-Data'
RAW_DIR = os.path.join(BASE_DIR, 'data', 'Raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'Processed')

print(f"Diretório Raw: {RAW_DIR}")
print(f"Diretório Processed: {PROCESSED_DIR}")

In [ ]:
# Listar arquivos disponíveis
print("Arquivos Raw:")
for f in os.listdir(RAW_DIR):
    print(f"  - {f}")

print("\nArquivos Processed:")
for f in os.listdir(PROCESSED_DIR):
    print(f"  - {f}")

---

## 2. Processamento do Índice de Gini

**Fonte**: IBGE - PNAD Contínua Anual

**Tabela SIDRA**: 7435

**Descrição**: Índice de Gini do rendimento domiciliar per capita (mede desigualdade de renda)

**Valores**: 0 (igualdade perfeita) a 1 (desigualdade máxima)

In [ ]:
# Ler arquivo raw do Gini
gini_raw_path = os.path.join(RAW_DIR, 'gini_sidra.csv')

# Visualizar estrutura do arquivo (primeiras linhas)
with open(gini_raw_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"Linha {i}: {line.strip()}")

In [ ]:
# Processar Índice de Gini
# O arquivo tem cabeçalho nas linhas 0-3, dados começam na linha 4
# IMPORTANTE: usar header=None para não perder a primeira UF (Rondônia)

gini_df = pd.read_csv(
    gini_raw_path,
    sep=';',
    skiprows=4,  # Pular cabeçalhos do SIDRA (linhas 0-3)
    header=None,  # Não usar primeira linha como header
    encoding='utf-8'
)

print("Colunas originais (índices):", gini_df.columns.tolist())
print(f"\nShape: {gini_df.shape}")
gini_df.head()

In [ ]:
# Renomear colunas e filtrar apenas UFs
# Estrutura: 0=Nível, 1=Cód, 2=UF, 3=2018, 4=2019, 5=2020, 6=2021, 7=2022

gini_df.columns = ['Nivel', 'Cod', 'UF', '2018', '2019', '2020', '2021', '2022']

# Filtrar apenas linhas onde Nivel = 'UF' (remove notas e fonte)
gini_df = gini_df[gini_df['Nivel'] == 'UF']

# Manter apenas UF e anos
gini_df = gini_df[['UF', '2018', '2019', '2020', '2021', '2022']]

print(f"UFs após filtro: {len(gini_df)}")
print("Colunas após renomear:")
gini_df.head()

In [ ]:
# Converter de formato wide para long (unpivot)
gini_long = gini_df.melt(
    id_vars=['UF'],
    value_vars=['2018', '2019', '2020', '2021', '2022'],
    var_name='Ano',
    value_name='Indice_Gini'
)

# Converter Ano para inteiro
gini_long['Ano'] = gini_long['Ano'].astype(int)

# Converter valores de string com vírgula para float
# Exemplo: "0,496" -> 0.496
gini_long['Indice_Gini'] = gini_long['Indice_Gini'].str.replace(',', '.').astype(float)

# Ordenar por UF e Ano
gini_long = gini_long.sort_values(['UF', 'Ano']).reset_index(drop=True)

print(f"Shape final: {gini_long.shape}")
print(f"Anos únicos: {sorted(gini_long['Ano'].unique())}")
print(f"UFs únicas: {gini_long['UF'].nunique()}")
gini_long.head(10)

In [ ]:
# Validação dos dados de Gini
print("=== Validação do Índice de Gini ===")
print(f"\nRegistros esperados: 135 (27 UFs x 5 anos)")
print(f"Registros obtidos: {len(gini_long)}")
print(f"\nValores nulos: {gini_long['Indice_Gini'].isna().sum()}")
print(f"\nEstatísticas descritivas:")
print(gini_long['Indice_Gini'].describe())

# Verificar range válido (0 a 1)
min_gini = gini_long['Indice_Gini'].min()
max_gini = gini_long['Indice_Gini'].max()
print(f"\nRange: {min_gini:.3f} - {max_gini:.3f}")
print(f"Range válido (0-1): {'OK' if 0 <= min_gini <= max_gini <= 1 else 'ERRO'}")

In [ ]:
# Salvar arquivo processado
gini_output_path = os.path.join(PROCESSED_DIR, 'gini_2018_2022.csv')
gini_long.to_csv(gini_output_path, index=False)

print(f"Arquivo salvo: {gini_output_path}")
print(f"Registros: {len(gini_long)}")

---

## 3. Processamento da Taxa de Gravidez Adolescente

**Fonte**: IBGE - Estatísticas do Registro Civil

**Tabela SIDRA**: 2609

**Descrição**: Percentual de nascidos vivos de mães adolescentes (<20 anos) em relação ao total

**Cálculo**: `(nascidos_menos_15 + nascidos_15_a_19) / total_nascidos * 100`

**Arquivos**:
- `nascidos_vivos_adolescentes_sidra.csv`: nascidos por faixa etária (<15 e 15-19 anos)
- `nascidos_vivos_total_sidra.csv`: total de nascidos vivos

In [ ]:
# Visualizar estrutura dos arquivos de nascidos vivos
adolescentes_path = os.path.join(RAW_DIR, 'nascidos_vivos_adolescentes_sidra.csv')
total_path = os.path.join(RAW_DIR, 'nascidos_vivos_total_sidra.csv')

print("=== Arquivo de Adolescentes (primeiras 10 linhas) ===")
with open(adolescentes_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"Linha {i}: {line.strip()}")

In [ ]:
print("=== Arquivo Total (primeiras 10 linhas) ===")
with open(total_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"Linha {i}: {line.strip()}")

In [ ]:
# Processar nascidos vivos de adolescentes
# IMPORTANTE: usar header=None para não perder a primeira UF

adolescentes_df = pd.read_csv(
    adolescentes_path,
    sep=';',
    skiprows=6,  # Pular cabeçalhos do SIDRA (6 linhas)
    header=None,  # Não usar primeira linha como header
    encoding='utf-8'
)

print("Colunas originais (índices):", adolescentes_df.columns.tolist())
print(f"Shape: {adolescentes_df.shape}")
adolescentes_df.head(3)

In [ ]:
# Renomear colunas do arquivo de adolescentes e filtrar UFs
# Estrutura: Nivel, Cod, UF, Sexo, 2018_menos15, 2018_15a19, 2019_menos15, 2019_15a19, ...

adolescentes_df.columns = [
    'Nivel', 'Cod', 'UF', 'Sexo',
    '2018_menos15', '2018_15a19',
    '2019_menos15', '2019_15a19',
    '2020_menos15', '2020_15a19',
    '2021_menos15', '2021_15a19',
    '2022_menos15', '2022_15a19'
]

# Filtrar apenas UFs
adolescentes_df = adolescentes_df[adolescentes_df['Nivel'] == 'UF']

# Manter apenas UF e colunas de dados
adolescentes_df = adolescentes_df[[
    'UF',
    '2018_menos15', '2018_15a19',
    '2019_menos15', '2019_15a19',
    '2020_menos15', '2020_15a19',
    '2021_menos15', '2021_15a19',
    '2022_menos15', '2022_15a19'
]]

print(f"UFs: {len(adolescentes_df)}")
adolescentes_df.head()

In [ ]:
# Processar total de nascidos vivos
total_df = pd.read_csv(
    total_path,
    sep=';',
    skiprows=6,  # Pular cabeçalhos do SIDRA
    header=None,  # Não usar primeira linha como header
    encoding='utf-8'
)

print("Colunas originais (índices):", total_df.columns.tolist())
print(f"Shape: {total_df.shape}")

# Renomear colunas e filtrar UFs
total_df.columns = ['Nivel', 'Cod', 'UF', 'Sexo', '2018', '2019', '2020', '2021', '2022']
total_df = total_df[total_df['Nivel'] == 'UF']

# Manter apenas UF e anos
total_df = total_df[['UF', '2018', '2019', '2020', '2021', '2022']]

print(f"UFs: {len(total_df)}")
total_df.head()

In [ ]:
# Calcular taxa de gravidez adolescente para cada ano
# Taxa = (menos_15 + 15_a_19) / total * 100

gravidez_data = []

for ano in ['2018', '2019', '2020', '2021', '2022']:
    for idx, row in adolescentes_df.iterrows():
        uf = row['UF']
        
        # Nascidos de mães adolescentes
        menos_15 = int(row[f'{ano}_menos15'])
        de_15_a_19 = int(row[f'{ano}_15a19'])
        total_adolescentes = menos_15 + de_15_a_19
        
        # Total de nascidos (buscar na outra tabela)
        total_nascidos = int(total_df[total_df['UF'] == uf][ano].values[0])
        
        # Calcular taxa percentual
        taxa = (total_adolescentes / total_nascidos) * 100
        
        gravidez_data.append({
            'UF': uf,
            'Ano': int(ano),
            'Nascidos_Adolescentes': total_adolescentes,
            'Nascidos_Total': total_nascidos,
            'Taxa_Gravidez_Adolescente': round(taxa, 2)
        })

gravidez_df = pd.DataFrame(gravidez_data)

print(f"Shape: {gravidez_df.shape}")
gravidez_df.head(10)

In [ ]:
# Validação dos dados de Gravidez Adolescente
print("=== Validação da Taxa de Gravidez Adolescente ===")
print(f"\nRegistros esperados: 135 (27 UFs x 5 anos)")
print(f"Registros obtidos: {len(gravidez_df)}")
print(f"\nValores nulos: {gravidez_df['Taxa_Gravidez_Adolescente'].isna().sum()}")
print(f"\nEstatísticas descritivas:")
print(gravidez_df['Taxa_Gravidez_Adolescente'].describe())

# UFs com maior taxa (possível indicador de vulnerabilidade)
print("\nTop 5 UFs com maior taxa média de gravidez adolescente:")
top_uf = gravidez_df.groupby('UF')['Taxa_Gravidez_Adolescente'].mean().sort_values(ascending=False).head(5)
print(top_uf)

In [ ]:
# Salvar apenas as colunas necessárias para o modelo
gravidez_final = gravidez_df[['UF', 'Ano', 'Taxa_Gravidez_Adolescente']]
gravidez_final = gravidez_final.sort_values(['UF', 'Ano']).reset_index(drop=True)

gravidez_output_path = os.path.join(PROCESSED_DIR, 'gravidez_adolescente_2018_2022.csv')
gravidez_final.to_csv(gravidez_output_path, index=False)

print(f"Arquivo salvo: {gravidez_output_path}")
print(f"Registros: {len(gravidez_final)}")

---

## 4. Processamento do PIB Total

**Fonte**: IBGE - Sistema de Contas Regionais

**Tabela SIDRA**: 5938

**Descrição**: Produto Interno Bruto a preços correntes

**Unidade**: Mil Reais

**Nota**: Optamos por usar PIB Total ao invés de PIB per capita, pois já temos Renda Per Capita como variável.

In [ ]:
# Ler arquivo raw do PIB
pib_raw_path = os.path.join(RAW_DIR, 'pib_sidra.csv')

# Visualizar estrutura
with open(pib_raw_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"Linha {i}: {line.strip()}")

In [ ]:
# Processar PIB
# IMPORTANTE: usar header=None para não perder a primeira UF
pib_df = pd.read_csv(
    pib_raw_path,
    sep=';',
    skiprows=4,  # Pular cabeçalhos do SIDRA
    header=None,  # Não usar primeira linha como header
    encoding='utf-8'
)

print("Colunas originais (índices):", pib_df.columns.tolist())
print(f"Shape: {pib_df.shape}")
pib_df.head()

In [ ]:
# Renomear colunas e filtrar UFs
pib_df.columns = ['Nivel', 'Cod', 'UF', '2018', '2019', '2020', '2021', '2022']
pib_df = pib_df[pib_df['Nivel'] == 'UF']

# Manter apenas UF e anos
pib_df = pib_df[['UF', '2018', '2019', '2020', '2021', '2022']]

print(f"UFs: {len(pib_df)}")
pib_df.head()

In [ ]:
# Converter de formato wide para long
pib_long = pib_df.melt(
    id_vars=['UF'],
    value_vars=['2018', '2019', '2020', '2021', '2022'],
    var_name='Ano',
    value_name='PIB_Total_MilReais'
)

# Converter tipos
pib_long['Ano'] = pib_long['Ano'].astype(int)
pib_long['PIB_Total_MilReais'] = pd.to_numeric(pib_long['PIB_Total_MilReais'], errors='coerce')

# Ordenar
pib_long = pib_long.sort_values(['UF', 'Ano']).reset_index(drop=True)

print(f"Shape final: {pib_long.shape}")
pib_long.head(10)

In [ ]:
# Validação dos dados de PIB
print("=== Validação do PIB Total ===")
print(f"\nRegistros esperados: 135 (27 UFs x 5 anos)")
print(f"Registros obtidos: {len(pib_long)}")
print(f"\nValores nulos: {pib_long['PIB_Total_MilReais'].isna().sum()}")
print(f"\nEstatísticas descritivas (em Mil R$):")
print(pib_long['PIB_Total_MilReais'].describe())

# Top 5 UFs por PIB médio
print("\nTop 5 UFs por PIB médio (Mil R$):")
top_pib = pib_long.groupby('UF')['PIB_Total_MilReais'].mean().sort_values(ascending=False).head(5)
for uf, pib in top_pib.items():
    print(f"  {uf}: R$ {pib:,.0f} mil")

In [ ]:
# Salvar arquivo processado
pib_output_path = os.path.join(PROCESSED_DIR, 'pib_2018_2022.csv')
pib_long.to_csv(pib_output_path, index=False)

print(f"Arquivo salvo: {pib_output_path}")
print(f"Registros: {len(pib_long)}")

---

## 5. Merge Final - Dataset Consolidado

Combinação de todas as variáveis no dataset final para modelagem.

**Dataset base**: `dados_modelo_final.csv` (já contém indicadores educacionais + IDHM + Desemprego + Renda)

**Variáveis a adicionar**:
- Índice de Gini
- Taxa de Gravidez Adolescente
- PIB Total

In [ ]:
# Carregar dataset base existente
base_path = os.path.join(PROCESSED_DIR, 'dados_modelo_final.csv')
df_base = pd.read_csv(base_path)

print("=== Dataset Base ===")
print(f"Shape: {df_base.shape}")
print(f"Colunas: {df_base.columns.tolist()}")
print(f"\nPrimeiros registros:")
df_base.head()

In [ ]:
# Carregar as 3 novas variáveis processadas
gini_final = pd.read_csv(os.path.join(PROCESSED_DIR, 'gini_2018_2022.csv'))
gravidez_final = pd.read_csv(os.path.join(PROCESSED_DIR, 'gravidez_adolescente_2018_2022.csv'))
pib_final = pd.read_csv(os.path.join(PROCESSED_DIR, 'pib_2018_2022.csv'))

print(f"Gini: {gini_final.shape}")
print(f"Gravidez: {gravidez_final.shape}")
print(f"PIB: {pib_final.shape}")

In [ ]:
# Garantir que a coluna Ano está no mesmo tipo em todos os dataframes
df_base['Ano'] = df_base['Ano'].astype(int)
gini_final['Ano'] = gini_final['Ano'].astype(int)
gravidez_final['Ano'] = gravidez_final['Ano'].astype(int)
pib_final['Ano'] = pib_final['Ano'].astype(int)

# Verificar UFs em cada dataset
print("UFs no dataset base:", sorted(df_base['UF'].unique())[:5], "...")
print("UFs no Gini:", sorted(gini_final['UF'].unique())[:5], "...")
print("UFs na Gravidez:", sorted(gravidez_final['UF'].unique())[:5], "...")
print("UFs no PIB:", sorted(pib_final['UF'].unique())[:5], "...")

In [ ]:
# Realizar merge das variáveis
# Merge 1: Base + Gini
df_merged = pd.merge(
    df_base,
    gini_final[['UF', 'Ano', 'Indice_Gini']],
    on=['UF', 'Ano'],
    how='left'
)
print(f"Após merge Gini: {df_merged.shape}")

# Merge 2: + Gravidez
df_merged = pd.merge(
    df_merged,
    gravidez_final[['UF', 'Ano', 'Taxa_Gravidez_Adolescente']],
    on=['UF', 'Ano'],
    how='left'
)
print(f"Após merge Gravidez: {df_merged.shape}")

# Merge 3: + PIB
df_merged = pd.merge(
    df_merged,
    pib_final[['UF', 'Ano', 'PIB_Total_MilReais']],
    on=['UF', 'Ano'],
    how='left'
)
print(f"Após merge PIB: {df_merged.shape}")

In [ ]:
# Validação do dataset final
print("=== Validação do Dataset Final ===")
print(f"\nShape: {df_merged.shape}")
print(f"\nColunas: {df_merged.columns.tolist()}")
print(f"\nValores nulos por coluna:")
print(df_merged.isna().sum())
print(f"\nTotal de valores nulos: {df_merged.isna().sum().sum()}")
print(f"\nTipos de dados:")
print(df_merged.dtypes)

In [ ]:
# Preview do dataset final
print("=== Preview do Dataset Final ===")
df_merged.head(10)

In [ ]:
# Estatísticas descritivas completas
print("=== Estatísticas Descritivas ===")
df_merged.describe()

In [ ]:
# Salvar dataset final (sobrescrever o existente)
output_path = os.path.join(PROCESSED_DIR, 'dados_modelo_final.csv')
df_merged.to_csv(output_path, index=False)

print(f"\n{'='*60}")
print(f"DATASET FINAL SALVO: {output_path}")
print(f"{'='*60}")
print(f"\nResumo:")
print(f"  - Registros: {len(df_merged)}")
print(f"  - Colunas: {len(df_merged.columns)}")
print(f"  - Variáveis preditoras: 6")
print(f"  - Variáveis target: 2 (Taxa_Abandono_Media, Taxa_Reprovacao_Media)")
print(f"  - Valores nulos: {df_merged.isna().sum().sum()}")
print(f"\nVariáveis preditoras:")
print(f"  1. IDHM")
print(f"  2. Taxa_Desemprego")
print(f"  3. Renda_Per_Capita")
print(f"  4. Indice_Gini (NOVA)")
print(f"  5. Taxa_Gravidez_Adolescente (NOVA)")
print(f"  6. PIB_Total_MilReais (NOVA)")

---

## 6. Resumo e Próximos Passos

### Dataset Final

| Aspecto | Valor |
|---------|-------|
| Arquivo | `data/Processed/dados_modelo_final.csv` |
| Registros | 135 (27 UFs × 5 anos) |
| Período | 2018-2022 |
| Variáveis preditoras | 6 |
| Variáveis target | 2 |
| Valores nulos | 0 |

### Variáveis do Modelo

**Targets (variáveis a prever)**:
- `Taxa_Abandono_Media`: Taxa média de abandono escolar (%)
- `Taxa_Reprovacao_Media`: Taxa média de reprovação escolar (%)

**Preditores (variáveis explicativas)**:
1. `IDHM`: Índice de Desenvolvimento Humano Municipal
2. `Taxa_Desemprego`: Taxa de desemprego (%)
3. `Renda_Per_Capita`: Renda per capita (R$)
4. `Indice_Gini`: Índice de desigualdade (0-1)
5. `Taxa_Gravidez_Adolescente`: % de nascidos de mães <20 anos
6. `PIB_Total_MilReais`: PIB estadual (Mil R$)

### Próximos Passos (CRISP-DM)

1. **Fase 4 - Modelagem**: Treinar modelos de ML (Random Forest, XGBoost, etc.)
2. **Fase 5 - Avaliação**: Validar modelos com métricas apropriadas
3. **Fase 6 - Implantação**: Documentar e disponibilizar resultados